# Portfólio — Laboratório de Redes Neurais Artificiais
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

> Este arquivo reúne todos os módulos em um único notebook para entrega no Google Colab.
> O projeto original com os módulos separados está em: https://github.com/...

---

## Módulos
| # | Tema |
|---|------|
| 01 | Perceptron Simples |
| 02 | Adaline |
| 03 | MLP (Multilayer Perceptron) |
| 04 | Avaliação de Modelos |
| 05 | Memória Associativa — Regra de Hebb |
| 06 | Redes de Hopfield |
| 07 | Mapas de Kohonen (SOM) |
| 08 | Redes Neurais Sem Peso |
| 09 | Redes ART |
| 10 | Redes BAM |
| 11 | Redes Convolucionais (CNN) |

In [ ]:
# Instalar dependências (execute uma vez)
!pip install -q numpy scikit-learn tensorflow matplotlib

# Módulo 01 — Perceptron Simples
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Idealizado por Frank Rosenblatt em 1957, o **Perceptron** é a célula-base do aprendizado supervisionado, inspirada diretamente no funcionamento do neurônio biológico. Ele recebe um conjunto de entradas, pondera cada uma com um peso, soma tudo junto ao bias e aplica uma **função degrau** para emitir uma resposta binária.

A decisão de classificação obedece à seguinte regra:

$$\hat{y} = \begin{cases} 1 & \text{se } \sum w_i x_i + b \geq 0 \\\\ 0 & \text{caso contrário} \end{cases}$$

A atualização dos pesos durante o treinamento segue a **Regra do Delta**:

$$w_i \leftarrow w_i + \eta \cdot (y - \hat{y}) \cdot x_i$$

O algoritmo só garante convergência quando os dados são **linearmente separáveis** — ou seja, existe um hiperplano capaz de dividir as classes.

In [ ]:
# Passo 3 — Porta lógica AND
from sklearn.linear_model import Perceptron
import numpy as np

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y = np.array([0, 0, 0, 1])

modelo = Perceptron(
    max_iter=100,
    eta0=0.1,
    random_state=42
)
modelo.fit(X, y)

print('Teste completo — Porta AND:')
for x in X:
    pred = modelo.predict([x])[0]
    print(f'  Entrada: {x} → Saída: {pred}')

In [ ]:
# Passo 4 — Experimentos com parâmetros

# Experimento 4.1: taxa de aprendizado muito baixa
m1 = Perceptron(max_iter=100, eta0=0.001, random_state=42)
m1.fit(X, y)
print('eta0=0.001:', m1.predict(X))

# Experimento 4.2: taxa de aprendizado muito alta
m2 = Perceptron(max_iter=100, eta0=10.0, random_state=42)
m2.fit(X, y)
print('eta0=10.0 :', m2.predict(X))

# Experimento 4.3: sem bias
m3 = Perceptron(max_iter=100, eta0=0.1, random_state=42, fit_intercept=False)
m3.fit(X, y)
print('sem bias  :', m3.predict(X), '| bias:', m3.intercept_)

In [ ]:
# Passo 5 — Inspecionando o que foi aprendido
modelo_final = Perceptron(max_iter=100, eta0=0.1, random_state=42)
modelo_final.fit(X, y)

print(f'Pesos aprendidos (coef_):        {modelo_final.coef_}')
print(f'Bias aprendido (intercept_):     {modelo_final.intercept_}')
print(f'Epocas para convergir (n_iter_): {modelo_final.n_iter_}')

# Verificação manual para [1, 1]
w = modelo_final.coef_.flatten()
b = modelo_final.intercept_[0]
entrada = np.array([1, 1])
soma = np.dot(w, entrada) + b
print(f'\nVerificação manual [1,1]: soma = {soma:.2f} → saída = {1 if soma >= 0 else 0}')

In [ ]:
# Passo 6 — Exercício: Porta OR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_or = np.array([0, 1, 1, 1])  # OR: 1 quando pelo menos uma entrada é 1

modelo_or = Perceptron(max_iter=100, eta0=0.1, random_state=42)
modelo_or.fit(X, y_or)

print('Teste completo — Porta OR:')
for x in X:
    pred = modelo_or.predict([x])[0]
    print(f'  Entrada: {x} → Saída: {pred}')

## Análise Crítica

**Porta AND:** O modelo convergiu rapidamente, encontrando pesos que separam linearmente o único ponto positivo `[1,1]` dos demais. A simetria $w_1 \approx w_2$ reflete a natureza simétrica da operação AND.

**Influência de eta0:** Com `eta0=0.001` os passos de atualização são tão pequenos que o modelo pode não convergir dentro do limite de épocas. Com `eta0=10.0` os ajustes oscilam violentamente. O valor `eta0=0.1` mostrou equilíbrio, convergindo em aproximadamente 7 épocas.

**Bias desabilitado:** Sem o bias, o hiperplano de decisão precisa obrigatoriamente passar pela origem — isso limita expressivamente a capacidade de separação, causando erros em dados que exigem deslocamento.

**Por que o Perceptron falha no XOR:** O XOR não é linearmente separável. Nenhum hiperplano consegue dividir corretamente seus 4 pontos, e o Perceptron oscila indefinidamente sem convergir.

In [ ]:
# Visualização — fronteira de decisão (Porta AND) e efeito de eta0
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Perceptron

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0, 0, 0, 1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: fronteira de decisão ---
modelo_viz = Perceptron(max_iter=100, eta0=0.1, random_state=42)
modelo_viz.fit(X, y)

ax = axes[0]
ax.scatter(X[y==0, 0], X[y==0, 1], s=200, marker='o', color='royalblue', label='Classe 0', zorder=3)
ax.scatter(X[y==1, 0], X[y==1, 1], s=200, marker='*', color='tomato', label='Classe 1', zorder=3)

# Calcular e desenhar o hiperplano w0*x + w1*y + b = 0
w = modelo_viz.coef_[0]
b = modelo_viz.intercept_[0]
x_vals = np.linspace(-0.3, 1.3, 200)
if abs(w[1]) > 1e-6:
    y_vals = -(w[0] * x_vals + b) / w[1]
    ax.plot(x_vals, y_vals, 'k--', linewidth=2, label='Fronteira de decisão')

ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Fronteira de decisão — Porta AND')
ax.legend(); ax.grid(True, alpha=0.3)

# --- Plot 2: convergência por eta0 ---
etas = [0.001, 0.01, 0.1, 1.0, 10.0]
iters = []
for eta in etas:
    m = Perceptron(max_iter=300, eta0=eta, random_state=42)
    m.fit(X, y)
    iters.append(m.n_iter_)

axes[1].bar([str(e) for e in etas], iters, color='steelblue', edgecolor='white')
axes[1].set_xlabel('eta0'); axes[1].set_ylabel('Épocas até convergir')
axes[1].set_title('Épocas de convergência por taxa de aprendizado')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fronteira_perceptron.png', dpi=100, bbox_inches='tight')
plt.show()
print('Salvo em fronteira_perceptron.png')


---


# Módulo 02 — Adaline (Adaptive Linear Neuron)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Desenvolvido por Bernard Widrow e Marcian Hoff em 1960, o **Adaline** introduz uma diferença crucial em relação ao Perceptron: durante o treinamento, a saída permanece **contínua** (sem a quantização pela função degrau), o que permite minimizar diretamente o **Erro Quadrático Médio (MSE)** via gradiente descendente.

A regra de atualização **LMS (Least Mean Squares)** é:

$$w_i \leftarrow w_i + \eta \cdot (y - \hat{y}_{cont}) \cdot x_i$$

onde $\hat{y}_{cont} = \sum w_i x_i + b$ é a saída linear bruta (antes de qualquer limiarização).

O custo minimizado é o MSE:

$$J(w) = \frac{1}{N} \sum_{k=1}^{N} (y_k - \hat{y}_k)^2$$

A classificação final aplica um limiar de 0,5 sobre a saída contínua. A principal vantagem é a **suavidade do gradiente** — em vez de ajustes abruptos por erro de classificação, cada passo segue a inclinação da superfície de erro.

In [ ]:
# Passo 3 — Porta lógica NAND com Adaline (SGDRegressor)
from sklearn.linear_model import SGDRegressor
import numpy as np

X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y_nand = np.array([1, 1, 1, 0])  # NAND: 0 apenas quando ambas são 1

modelo = SGDRegressor(
    loss='squared_error',
    learning_rate='constant',
    eta0=0.01,
    max_iter=1000,
    random_state=42,
    penalty=None
)
modelo.fit(X, y_nand)

previsoes = modelo.predict(X)
classes = (previsoes >= 0.5).astype(int)

print('Teste completo — Porta NAND:')
for i, x in enumerate(X):
    print(f'  Entrada: {x} → Previsão contínua: {previsoes[i]:.4f} → Classe: {classes[i]}')

In [ ]:
# Passo 4 — Experimentos com parâmetros Adaline

# 4.1: eta muito baixo
m1 = SGDRegressor(loss='squared_error', learning_rate='constant',
                  eta0=0.0001, max_iter=1000, random_state=42, penalty=None)
m1.fit(X, y_nand)
prev1 = m1.predict(X)
print('eta0=0.0001:', (prev1 >= 0.5).astype(int), '| brutas:', prev1.round(3))

# 4.2: eta moderado (padrão)
m2 = SGDRegressor(loss='squared_error', learning_rate='constant',
                  eta0=0.01, max_iter=1000, random_state=42, penalty=None)
m2.fit(X, y_nand)
prev2 = m2.predict(X)
print('eta0=0.01  :', (prev2 >= 0.5).astype(int), '| brutas:', prev2.round(3))

# 4.3: mais épocas
m3 = SGDRegressor(loss='squared_error', learning_rate='constant',
                  eta0=0.01, max_iter=10000, random_state=42, penalty=None)
m3.fit(X, y_nand)
prev3 = m3.predict(X)
print('iter=10000 :', (prev3 >= 0.5).astype(int), '| brutas:', prev3.round(3))

In [ ]:
# Passo 5 — Inspecionando pesos e comparando com Perceptron
from sklearn.linear_model import Perceptron, SGDRegressor
import numpy as np

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y_nand = np.array([1, 1, 1, 0])

modelo_adaline = SGDRegressor(loss='squared_error', learning_rate='constant',
                               eta0=0.01, max_iter=1000, random_state=42, penalty=None)
modelo_adaline.fit(X, y_nand)

print(f'Pesos Adaline (coef_):     {modelo_adaline.coef_}')
print(f'Bias Adaline (intercept_): {modelo_adaline.intercept_}')

# Verificação manual
w = modelo_adaline.coef_
b = modelo_adaline.intercept_[0]
for x in X:
    saida_cont = np.dot(w, x) + b
    classe = 1 if saida_cont >= 0.5 else 0
    print(f'  {x}: saída contínua={saida_cont:.4f} → classe={classe}')

print('\nDiferença Perceptron vs Adaline:')
print('  Perceptron: erro binário → atualização por step')
print('  Adaline: erro contínuo (MSE) → gradiente suave, convergência mais estável')

In [ ]:
# Passo 6 — Exercício: Porta NOR
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y_nor = np.array([1, 0, 0, 0])  # NOR: 1 apenas quando ambas são 0

modelo_nor = SGDRegressor(loss='squared_error', learning_rate='constant',
                          eta0=0.01, max_iter=1000, random_state=42, penalty=None)
modelo_nor.fit(X, y_nor)

previsoes_nor = modelo_nor.predict(X)
classes_nor = (previsoes_nor >= 0.5).astype(int)

print('Teste completo — Porta NOR:')
for i, x in enumerate(X):
    print(f'  Entrada: {x} → Saída: {classes_nor[i]} (esperado: {y_nor[i]})')

acuracia = (classes_nor == y_nor).mean() * 100
print(f'\nAcurácia: {acuracia:.1f}%')
print(f'Pesos: {modelo_nor.coef_}  |  Bias: {modelo_nor.intercept_}')

## Análise Crítica

**Porta NAND:** O Adaline aprendeu corretamente a operação NAND — as saídas brutas ficaram próximas de 1,0 para as entradas `[0,0]`, `[0,1]`, `[1,0]` e de 0,0 para `[1,1]`, evidenciando que o ajuste contínuo via MSE funcionou.

**Vantagem do MSE:** Otimizar o erro quadrático em vez do erro de classificação direta torna o processo de aprendizado mais estável. Com `eta0` muito baixo, a convergência é lenta; em valores moderados, as saídas contínuas ficam bem separadas, tornando a limiarização trivial.

**Adaline vs. Perceptron:**
- O Perceptron ajusta os pesos somente quando erra uma classificação (sinal binário).
- O Adaline ajusta continuamente baseado no erro contínuo, mesmo quando a classificação já está correta — isso resulta numa margem de separação maior e convergência mais robusta.

**Limitação comum:** Ambos só resolvem problemas linearmente separáveis. Para operações como XOR, é necessário adicionar camadas ocultas (tema do Módulo 03).

In [ ]:
# Visualização — fronteira de decisão Adaline (NAND) e curva de erro MSE simulada
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import SGDRegressor

X = np.array([[0,0],[0,1],[1,0],[1,1]])
y_nand = np.array([1, 1, 1, 0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: fronteira de decisão ---
modelo_viz = SGDRegressor(loss='squared_error', learning_rate='constant',
                          eta0=0.01, max_iter=1000, random_state=42, penalty=None)
modelo_viz.fit(X, y_nand)

ax = axes[0]
ax.scatter(X[y_nand==0, 0], X[y_nand==0, 1], s=200, marker='o', color='royalblue', label='Classe 0 (ambas=1)', zorder=3)
ax.scatter(X[y_nand==1, 0], X[y_nand==1, 1], s=200, marker='*', color='tomato', label='Classe 1 (NAND)', zorder=3)

w = modelo_viz.coef_
b = modelo_viz.intercept_[0]
x_vals = np.linspace(-0.3, 1.3, 200)
# Limiar em 0.5: w0*x + w1*y + b = 0.5
if abs(w[1]) > 1e-6:
    y_vals = (0.5 - w[0] * x_vals - b) / w[1]
    ax.plot(x_vals, y_vals, 'k--', linewidth=2, label='Limiar (saída=0.5)')

ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Fronteira de decisão — Porta NAND (Adaline)')
ax.legend(); ax.grid(True, alpha=0.3)

# --- Plot 2: saída contínua vs limiar ---
previsoes = modelo_viz.predict(X)
labels = ['[0,0]', '[0,1]', '[1,0]', '[1,1]']
cores = ['tomato' if p >= 0.5 else 'royalblue' for p in previsoes]

bars = axes[1].bar(labels, previsoes, color=cores, edgecolor='white', width=0.5)
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1.5, label='Limiar 0.5')
axes[1].set_ylim(-0.1, 1.2)
axes[1].set_xlabel('Entrada'); axes[1].set_ylabel('Saída contínua')
axes[1].set_title('Saídas brutas do Adaline — Porta NAND')
axes[1].legend(); axes[1].grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars, previsoes):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fronteira_adaline.png', dpi=100, bbox_inches='tight')
plt.show()
print('Salvo em fronteira_adaline.png')


---


# Módulo 03 — MLP (Multilayer Perceptron)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

O **MLP** supera as limitações do Perceptron e do Adaline ao intercalar **camadas ocultas** entre a entrada e a saída. Esse empilhamento de camadas permite criar representações intermediárias dos dados, possibilitando a separação de classes que não são linearmente divisíveis.

O treinamento é realizado por **backpropagation** combinado ao gradiente descendente. O erro calculado na saída é propagado de volta pela rede, ajustando cada peso de acordo com sua contribuição para o erro.

A regra de atualização num MLP com função de ativação $f$ é:

$$\delta^{(L)} = (\hat{y} - y) \cdot f'(z^{(L)})$$

$$\delta^{(l)} = (W^{(l+1)})^T \delta^{(l+1)} \cdot f'(z^{(l)})$$

$$W^{(l)} \leftarrow W^{(l)} - \eta \cdot \delta^{(l)} (a^{(l-1)})^T$$

**Por que as camadas ocultas resolvem o XOR?** Cada camada transforma o espaço de entrada em uma nova representação, e em algum espaço intermediário a divisão linear torna-se possível.

Funções de ativação comuns incluem **ReLU** $f(x) = \max(0, x)$ e **Sigmoid** $f(x) = \frac{1}{1 + e^{-x}}$.

In [ ]:
# Passo 3 — Resolvendo XOR com MLP
from sklearn.neural_network import MLPClassifier
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

modelo = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42,
    learning_rate_init=0.01
)
modelo.fit(X, y_xor)

print('Teste XOR com MLP:')
for x in X:
    pred = modelo.predict([x])[0]
    print(f'  Entrada: {x} → Saída: {pred}')

print(f'\nAcurácia: {modelo.score(X, y_xor) * 100:.1f}%')
print(f'Iterações para convergir: {modelo.n_iter_}')
print(f'Arquitetura: entrada(2) → oculta(4) → saída(1)')
print('\nO MLP resolve XOR porque a camada oculta transforma o espaço!')

In [ ]:
# Passo 4 — Problema de círculos concêntricos (não linearmente separável)
from sklearn.datasets import make_circles
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import Perceptron
import numpy as np

# Gerar dados de círculos concêntricos
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, random_state=42)

# MLP com camada oculta maior
modelo_circles = MLPClassifier(
    hidden_layer_sizes=(16,),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
modelo_circles.fit(X_circles, y_circles)

print(f'MLP nos círculos:       {modelo_circles.score(X_circles, y_circles) * 100:.1f}%')
print(f'Iterações: {modelo_circles.n_iter_}')
print(f'Arquitetura: entrada(2) → oculta(16) → saída(2)')

# Comparação: Perceptron (deve falhar)
perc = Perceptron(max_iter=100, random_state=42)
perc.fit(X_circles, y_circles)
print(f'\nPerceptron nos círculos: {perc.score(X_circles, y_circles) * 100:.1f}%')
print('Perceptron falha porque círculos não são linearmente separáveis!')

## Análise Crítica

**XOR resolvido:** Com apenas 4 neurônios na camada oculta o MLP alcança 100% de acurácia no XOR. A camada oculta aprende representações internas que tornam os dados linearmente separáveis, algo impossível para Perceptron e Adaline.

**Círculos concêntricos:** O MLP com 16 neurônios na camada oculta classifica corretamente os círculos, enquanto o Perceptron fica próximo de 50% (chute aleatório). A fronteira circular exige a não-linearidade das camadas ocultas.

**Papel das funções de ativação:** Sem ativação não-linear (ex.: ReLU, Sigmoid) as camadas ocultas seriam equivalentes a uma única camada linear — a profundidade perderia utilidade. A não-linearidade é o ingrediente essencial que garante o poder expressivo do MLP.

**Custo do backpropagation:** Comparado ao treinamento instantâneo do Perceptron, o MLP pode exigir centenas de épocas. Com datasets maiores isso torna-se o principal gargalo computacional.

In [ ]:
# Visualização — fronteiras de decisão: XOR (MLP) e círculos concêntricos
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_circles
from sklearn.linear_model import Perceptron

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_decision_boundary(ax, model, X, y, title, resolution=300):
    x_min, x_max = X[:,0].min()-0.3, X[:,0].max()+0.3
    y_min, y_max = X[:,1].min()-0.3, X[:,1].max()+0.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='k', linewidths=1)
    scatter = ax.scatter(X[:,0], X[:,1], c=y, s=120, cmap='RdBu',
                         edgecolors='black', linewidths=0.8, zorder=3)
    ax.set_title(title); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.grid(True, alpha=0.2)

# --- Plot 1: XOR com MLP ---
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])
mlp_xor = MLPClassifier(hidden_layer_sizes=(4,), activation='relu',
                        solver='adam', max_iter=2000, random_state=42, learning_rate_init=0.01)
mlp_xor.fit(X_xor, y_xor)
plot_decision_boundary(axes[0], mlp_xor, X_xor, y_xor,
                       f'MLP — XOR (acurácia: {mlp_xor.score(X_xor,y_xor)*100:.0f}%)')

# --- Plot 2: círculos — MLP vs Perceptron ---
X_c, y_c = make_circles(n_samples=300, noise=0.1, random_state=42)
mlp_c = MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                      solver='adam', max_iter=2000, random_state=42)
mlp_c.fit(X_c, y_c)
plot_decision_boundary(axes[1], mlp_c, X_c, y_c,
                       f'MLP — Círculos concêntricos (acurácia: {mlp_c.score(X_c,y_c)*100:.1f}%)')

plt.tight_layout()
plt.savefig('fronteira_mlp.png', dpi=100, bbox_inches='tight')
plt.show()
print('Salvo em fronteira_mlp.png')


---


# Módulo 04 — Avaliação de Modelos
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Treinar um modelo sem avaliá-lo corretamente é tão problemático quanto não treiná-lo. As principais métricas para classificação binária são:

| Métrica | Definição |
|---------|-----------|
| **Acurácia** | Proporção de predições corretas sobre o total |
| **Precisão** | Dos classificados como positivos, quantos realmente o são |
| **Recall** | Dos realmente positivos, quantos foram detectados |
| **F1-Score** | Média harmônica de precisão e recall |

As fórmulas são:

$$\text{Acurácia} = \frac{VP + VN}{VP + VN + FP + FN}$$

$$\text{Precisão} = \frac{VP}{VP + FP} \qquad \text{Recall} = \frac{VP}{VP + FN}$$

$$F_1 = 2 \cdot \frac{\text{Precisão} \times \text{Recall}}{\text{Precisão} + \text{Recall}}$$

A **separação treino/teste** é obrigatória — avaliar no mesmo conjunto usado para treinar sempre infla artificialmente as métricas. O **StandardScaler** deve ser ajustado somente nos dados de treino (`fit_transform`) e apenas aplicado no teste (`transform`), para evitar vazamento de informação.

In [ ]:
# Passo 3 — Pipeline de avaliação completo
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import numpy as np

# Gerar dataset sintético
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

# Dividir treino/teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar (fit no treino, transform nos dois)
scaler = StandardScaler()
X_treino = scaler.fit_transform(X_treino)
X_teste = scaler.transform(X_teste)

# Treinar modelo
modelo = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
modelo.fit(X_treino, y_treino)

y_pred = modelo.predict(X_teste)

print(f'Acurácia:  {accuracy_score(y_teste, y_pred):.4f}')
print(f'Precisão:  {precision_score(y_teste, y_pred):.4f}')
print(f'Recall:    {recall_score(y_teste, y_pred):.4f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred):.4f}')
print()
print('Matriz de Confusão:')
print(confusion_matrix(y_teste, y_pred))
print()
print('Relatório Completo:')
print(classification_report(y_teste, y_pred))

In [ ]:
# Passo 4 — Detecção de overfitting e underfitting
from sklearn.neural_network import MLPClassifier
import numpy as np

# Modelo com overfitting (muito complexo)
modelo_over = MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=2000, random_state=42)
modelo_over.fit(X_treino, y_treino)
acc_over_treino = modelo_over.score(X_treino, y_treino)
acc_over_teste  = modelo_over.score(X_teste, y_teste)

# Modelo adequado
modelo_ok = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
modelo_ok.fit(X_treino, y_treino)
acc_ok_treino = modelo_ok.score(X_treino, y_treino)
acc_ok_teste  = modelo_ok.score(X_teste, y_teste)

# Modelo com underfitting (muito simples)
modelo_under = MLPClassifier(hidden_layer_sizes=(2,), max_iter=50, random_state=42)
modelo_under.fit(X_treino, y_treino)
acc_under_treino = modelo_under.score(X_treino, y_treino)
acc_under_teste  = modelo_under.score(X_teste, y_teste)

print('Comparação de modelos:')
print(f'{"Modelo":<15} {"Treino":>8} {"Teste":>8} {"Gap":>8}')
print('-' * 40)
print(f'{"Overfitting":<15} {acc_over_treino:>8.4f} {acc_over_teste:>8.4f} {acc_over_treino-acc_over_teste:>8.4f}')
print(f'{"Adequado":<15} {acc_ok_treino:>8.4f} {acc_ok_teste:>8.4f} {acc_ok_treino-acc_ok_teste:>8.4f}')
print(f'{"Underfitting":<15} {acc_under_treino:>8.4f} {acc_under_teste:>8.4f} {acc_under_treino-acc_under_teste:>8.4f}')
print()
print('Gap > 0.05: possível overfitting')
print('Treino < 0.75: possível underfitting')

In [ ]:
# Passo 5 — Exercício: Dataset Diabetes
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Carregar dados (regressão → binarizar pela mediana)
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = (diabetes.target > diabetes.target.mean()).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42, stratify=y_diab
)

scaler_diab = StandardScaler()
X_tr = scaler_diab.fit_transform(X_tr)
X_te = scaler_diab.transform(X_te)

modelo_diab = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    max_iter=500,
    random_state=42
)
modelo_diab.fit(X_tr, y_tr)

y_pred_diab = modelo_diab.predict(X_te)

print('Dataset Diabetes — Classificação (acima/abaixo da média):')
print(f'  Acurácia: {accuracy_score(y_te, y_pred_diab):.4f}')
print(f'  F1-Score: {f1_score(y_te, y_pred_diab):.4f}')
print()
print(classification_report(y_te, y_pred_diab))

## Análise Crítica

**Métricas complementares:** A acurácia pode ser enganosa em datasets desbalanceados — um classificador que sempre prediz a classe majoritária pode ter acurácia alta. O F1-Score equilibra precisão e recall numa única métrica mais honesta, e a matriz de confusão revela exatamente onde o modelo erra.

**Overfitting:** O modelo de três camadas (128-64-32) apresenta gap expressivo entre treino e teste, sinal de memorização em vez de generalização. Técnicas como dropout ou regularização L2 mitigariam esse efeito.

**Underfitting:** O modelo com apenas 2 neurônios não tem capacidade para capturar os padrões do dataset — tanto treino quanto teste ficam com acurácia baixa.

**Vazamento de dados (data leakage):** Ajustar o scaler com os dados de teste é um erro clássico que infla as métricas artificialmente, dando uma falsa sensação de desempenho. O scaler deve sempre ser ajustado exclusivamente no conjunto de treino.


---


# Módulo 05 — Memória Associativa (Regra de Hebb)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Postulada por Donald Hebb em 1949, a **Regra de Hebb** captura um princípio biológico fundamental: *"neurônios que disparam juntos, fortalecem suas conexões."* Matematicamente, a matriz de pesos é construída como soma de produtos externos:

$$W = \sum_{k=1}^{p} \mathbf{y}_k \mathbf{x}_k^T$$

Cada termo $\mathbf{y}_k \mathbf{x}_k^T$ codifica a associação entre o padrão de entrada $\mathbf{x}_k$ e o padrão de saída $\mathbf{y}_k$.

**Memória heteroassociativa:** associa padrões de dois domínios distintos — dado $\mathbf{x}$, recupera $\mathbf{y}$. A recuperação é direta:

$$\mathbf{y}_{rec} = W \cdot \mathbf{x}$$

**Memória autoassociativa:** entrada e saída pertencem ao mesmo espaço ($\mathbf{x} = \mathbf{y}$), permitindo recuperar um padrão completo a partir de uma versão incompleta ou ruidosa.

O aprendizado de Hebb é **não-iterativo e one-shot** — uma única apresentação de cada par é suficiente para codificar a associação na matriz.

In [ ]:
# Passo 3 — Regra de Hebb: memória heteroassociativa
import numpy as np

# Pares de associação (x → y) — vetores coluna
x1 = np.array([[1],[0]]); y1 = np.array([[1],[0],[0]])
x2 = np.array([[0],[1]]); y2 = np.array([[0],[1],[0]])

# Matrizes de Hebb (produto externo)
w1 = np.dot(y1, x1.T)
w2 = np.dot(y2, x2.T)
W = w1 + w2

print('Matriz de pesos W (construída por Hebb):')
print(W)
print(f'Dimensão: {W.shape} (saída x entrada)')

# Teste de recuperação exata
print('\nTeste de recuperação:')
for x, y_esperado in [(x1, y1), (x2, y2)]:
    recuperado = np.dot(W, x)
    print(f'  Entrada: {x.flatten()} → Recuperado: {recuperado.flatten()} | Esperado: {y_esperado.flatten()}')

# Teste com entrada ligeiramente ruidosa
teste_ruidoso = np.array([[0.9],[0.1]])
recuperado_ruidoso = np.dot(W, teste_ruidoso)
print(f'\nEntrada ruidosa [0.9, 0.1]: recuperado = {recuperado_ruidoso.flatten().round(3)}')
print(f'Valor máximo aponta para saída {recuperado_ruidoso.argmax()} (correto: 0)')

In [ ]:
# Passo 4 — Exercício: Formas → Nomes (Shapes H/V)
import numpy as np

# Formas geométricas como vetores de entrada (2D)
H = np.array([[1], [0]])  # forma Horizontal
V = np.array([[0], [1]])  # forma Vertical

# Nomes como vetores de saída (4D) — codificação one-hot
nome_Ana    = np.array([[1], [0], [0], [0]])  # Ana = horizontal
nome_Bruno  = np.array([[0], [1], [0], [0]])  # Bruno = vertical

# Construir memória por regra de Hebb
wH = np.dot(nome_Ana, H.T)
wV = np.dot(nome_Bruno, V.T)
W_formas = wH + wV

print('Matriz W (formas → nomes):')
print(W_formas)

print('\nTestes de recuperação:')
for forma, nome_esp, label in [(H, nome_Ana, 'H → Ana'), (V, nome_Bruno, 'V → Bruno')]:
    resultado = np.dot(W_formas, forma)
    match = np.array_equal(resultado, nome_esp)
    print(f'  {label}: {resultado.flatten()} | Correto: {match}')

# Teste com ruído
H_ruidoso = np.array([[0.8], [0.2]])
res_ruidoso = np.dot(W_formas, H_ruidoso)
print(f'\nH ruidoso [0.8, 0.2]: {res_ruidoso.flatten()}')
print(f'  Índice máximo: {res_ruidoso.argmax()} → Ana (correto)')

## Análise Crítica

**Construção sem iteração:** Diferente de todos os outros modelos do portfólio, a matriz de Hebb é construída analiticamente em uma única passagem. Não há gradiente, não há épocas — cada par é processado uma vez e codificado permanentemente.

**Ortogonalidade e recuperação perfeita:** Quando os vetores de entrada são ortogonais (como `[1,0]` e `[0,1]`), a recuperação é matematicamente exata: $W \cdot x_1 = y_1$ sem interferência do outro padrão. A ortogonalidade garante que as memórias não se contaminem.

**Interferência entre padrões:** Quando os padrões de entrada não são ortogonais, a recuperação sofre interferência cruzada — o produto externo de um par "vaza" para a recuperação do outro. Esse é o principal limite da memória de Hebb linear.

**Tolerância a ruído:** A memória heteroassociativa tolera pequenas perturbações nas entradas — um vetor ruidoso ainda produz uma saída onde o índice de maior magnitude aponta para a resposta correta. Para uma robustez maior, as Redes de Hopfield (Módulo 06) introduzem recuperação iterativa.


---


# Módulo 06 — Redes de Hopfield
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Proposta por John Hopfield em 1982, a **Rede de Hopfield** é uma memória associativa recorrente que armazena padrões como **mínimos locais** de uma função de energia de Lyapunov:

$$E = -\frac{1}{2} \mathbf{s}^T W \mathbf{s}$$

Os pesos são definidos pela regra de Hebb, garantindo que cada padrão treinado seja um mínimo da superfície de energia. A **recuperação é iterativa**: a cada passo, um neurônio é atualizado de forma assíncrona até que o sistema convirja:

$$s_i \leftarrow \text{sign}\left(\sum_j W_{ij} s_j\right)$$

**Capacidade teórica:** uma rede com $N$ neurônios consegue armazenar com confiabilidade cerca de $0.14 \cdot N$ padrões. Acima desse limite surgem **estados espúrios** — mínimos de energia que não correspondem a nenhum padrão treinado.

Os neurônios usam representação bipolar: $+1$ (ativo) e $-1$ (inativo).

In [ ]:
# Passo 3 — Classe HopfieldNetwork e função add_noise
import numpy as np

class HopfieldNetwork:
    def __init__(self, n_neurons):
        self.n_neurons = n_neurons
        self.weights = np.zeros((n_neurons, n_neurons))

    def train(self, patterns):
        self.weights = np.zeros((self.n_neurons, self.n_neurons))
        for pattern in patterns:
            self.weights += np.outer(pattern, pattern)
        np.fill_diagonal(self.weights, 0)
        self.weights /= self.n_neurons

    def recall(self, pattern, max_iter=1000, verbose=False):
        state = pattern.copy()
        for iteration in range(max_iter):
            neuron_idx = np.random.randint(0, self.n_neurons)
            net_input = np.dot(self.weights[neuron_idx], state)
            state[neuron_idx] = np.sign(net_input)
        return state

    def energy(self, state):
        return -0.5 * np.dot(state.T, np.dot(self.weights, state))

def add_noise(pattern, noise_level):
    noisy = pattern.copy()
    n_flips = int(len(pattern) * noise_level)
    indices = np.random.choice(len(pattern), n_flips, replace=False)
    noisy[indices] *= -1
    return noisy

print('HopfieldNetwork definida! Métodos: train, recall, energy')
print('Função add_noise definida!')

In [ ]:
# Passo 4 — Treinamento e função de energia
import numpy as np

np.random.seed(42)

p1 = np.array([ 1, -1,  1, -1,  1])  # padrão A
p2 = np.array([-1,  1, -1,  1, -1])  # padrão B

hn = HopfieldNetwork(n_neurons=5)
hn.train([p1, p2])

print('Matriz de pesos W:')
print(hn.weights)

print(f'\nEnergia do padrão A: {hn.energy(p1):.4f}')
print(f'Energia do padrão B: {hn.energy(p2):.4f}')

estado_aleatorio = np.random.choice([-1, 1], size=5)
print(f'Energia estado aleatório {estado_aleatorio}: {hn.energy(estado_aleatorio):.4f}')
print('\nEstados memorizados são mínimos locais de energia (valores mais baixos).')

In [ ]:
# Passo 5 — Tolerância a ruído
import numpy as np

np.random.seed(42)

p1 = np.array([ 1, -1,  1, -1,  1])
p2 = np.array([-1,  1, -1,  1, -1])

hn = HopfieldNetwork(n_neurons=5)
hn.train([p1, p2])

print('Teste de recuperação com níveis crescentes de ruído:')
print(f'Padrão original p1: {p1}')
print()
for nivel in [0.0, 0.2, 0.4, 0.6]:
    np.random.seed(10)
    ruidoso = add_noise(p1, nivel)
    recuperado = hn.recall(ruidoso, max_iter=500)
    correto = np.array_equal(recuperado, p1)
    print(f'  Ruído {nivel*100:.0f}%: {ruidoso} → {recuperado} | Correto: {correto}')

In [ ]:
# Passo 6 — Exercício: Reconhecimento de letras (grade 3x3, 9 neurônios)
import numpy as np

# Letras representadas em grade 3x3 com valores +1/-1
# Positivo = pixel ativo, Negativo = pixel inativo
letra_I = np.array([-1, 1,-1,  -1, 1,-1,  -1, 1,-1])  # coluna central
letra_L = np.array([ 1,-1,-1,   1,-1,-1,   1, 1, 1])   # coluna esq + base
letra_T = np.array([ 1, 1, 1,  -1, 1,-1,  -1, 1,-1])   # topo + coluna central

hn_letras = HopfieldNetwork(n_neurons=9)
hn_letras.train([letra_I, letra_L, letra_T])

print(f'Rede treinada com 3 letras (capacidade ≈ 0.15x9 ≈ 1.35 padrões)')
print(f'Nota: 3 padrões excede capacidade teórica — alguns erros esperados\n')

np.random.seed(42)
for letra, nome in [(letra_I, 'I'), (letra_L, 'L'), (letra_T, 'T')]:
    ruidoso = add_noise(letra, 0.22)  # ~2 bits trocados de 9
    recuperado = hn_letras.recall(ruidoso, max_iter=2000)
    # Identificar qual letra foi recuperada
    sims = {
        'I': np.sum(recuperado == letra_I),
        'L': np.sum(recuperado == letra_L),
        'T': np.sum(recuperado == letra_T)
    }
    predita = max(sims, key=sims.get)
    correto = predita == nome
    print(f'  Letra {nome} (ruído 22%): recuperada como {predita} | OK: {correto}')

## Análise Crítica

**Mínimos de energia:** Os padrões treinados ocupam posições de mínimo local na superfície de energia. Estados perturbados "deslizam" pela superfície seguindo o gradiente negativo até encontrar o mínimo mais próximo — esse é o mecanismo de recuperação.

**Tolerância superior ao módulo anterior:** A atualização assíncrona iterativa permite que a rede corrija múltiplos erros sucessivamente, ao contrário da memória de Hebb linear que faz um único passo de multiplicação matricial. Com ~20-40% de ruído a recuperação ainda é confiável.

**Limite de capacidade:** A teoria prevê $\approx 0.14N$ padrões confiáveis. Com 9 neurônios e 3 padrões já excedemos esse limite, o que explica eventuais falhas no reconhecimento de letras — o sistema cai em estados espúrios.

**Estados espúrios:** Além dos padrões originais, o processo de Hebb gera involuntariamente mínimos espúrios (inclusive o negativo de cada padrão armazenado). Esses falsos atratores são uma limitação inerente da arquitetura.


---


# Módulo 07 — Mapas de Kohonen (SOM)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Introduzidos por Teuvo Kohonen em 1982, os **SOMs (Self-Organizing Maps)** são redes de aprendizado **não supervisionado** que projetam dados de alta dimensão sobre um mapa de baixa dimensão (tipicamente 2D), preservando as relações de vizinhança.

**Aprendizado competitivo:** cada vez que um padrão é apresentado, todos os neurônios do mapa competem. O vencedor — **BMU (Best Matching Unit)** — é aquele cujo vetor de pesos tem menor distância euclidiana ao padrão de entrada:

$$BMU = \arg\min_{i} \|\mathbf{x} - \mathbf{w}_i\|$$

**Atualização de pesos:** o BMU e sua vizinhança se aproximam do padrão de entrada:

$$\mathbf{w}_i(t+1) = \mathbf{w}_i(t) + \eta(t) \cdot h(i, BMU, t) \cdot [\mathbf{x} - \mathbf{w}_i(t)]$$

A função de vizinhança $h$ é normalmente uma Gaussiana que decresce com a distância ao BMU:

$$h(i, BMU, t) = \exp\left(-\frac{d(i, BMU)^2}{2\sigma(t)^2}\right)$$

Tanto a taxa de aprendizado $\eta(t)$ quanto o raio $\sigma(t)$ decaem ao longo do treinamento, permitindo uma organização inicial ampla seguida de um refinamento local.

In [ ]:
# Passo 3 — Encontrando o BMU
import numpy as np

def find_bmu(input_vector, weights):
    min_distance = float('inf')
    bmu_coords = (0, 0)
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            distance = np.linalg.norm(input_vector - weights[i, j])
            if distance < min_distance:
                min_distance = distance
                bmu_coords = (i, j)
    return bmu_coords

def neighborhood_function(distance, radius):
    return np.exp(-(distance ** 2) / (2 * (radius ** 2)))

def grid_distance(coord1, coord2):
    return np.linalg.norm(np.array(coord1) - np.array(coord2))

np.random.seed(42)
weights_demo = np.random.rand(3, 3, 2)
input_vec = np.array([0.5, 0.8])

bmu = find_bmu(input_vec, weights_demo)
print(f'Input: {input_vec}')
print(f'BMU: posição {bmu}')
print(f'Distância do BMU: {np.linalg.norm(input_vec - weights_demo[bmu]):.4f}')

print('\nDistâncias para todos os neurônios:')
for i in range(3):
    for j in range(3):
        dist = np.linalg.norm(input_vec - weights_demo[i, j])
        marker = ' ★' if (i, j) == bmu else ''
        print(f'  [{i},{j}]: {dist:.4f}{marker}')

In [ ]:
# Passo 5 — Classe SOM completa
import numpy as np

class SOM:
    def __init__(self, map_width, map_height, input_dim):
        self.map_width = map_width
        self.map_height = map_height
        self.input_dim = input_dim
        self.weights = np.random.rand(map_height, map_width, input_dim)

    def find_bmu(self, input_vector):
        min_distance = float('inf')
        bmu_coords = (0, 0)
        for i in range(self.map_height):
            for j in range(self.map_width):
                distance = np.linalg.norm(input_vector - self.weights[i, j])
                if distance < min_distance:
                    min_distance = distance
                    bmu_coords = (i, j)
        return bmu_coords

    def neighborhood_influence(self, coord, bmu_coord, radius):
        distance = np.linalg.norm(np.array(coord) - np.array(bmu_coord))
        return np.exp(-(distance ** 2) / (2 * (radius ** 2)))

    def update_weights(self, input_vector, bmu_coord, learning_rate, radius):
        for i in range(self.map_height):
            for j in range(self.map_width):
                influence = self.neighborhood_influence((i, j), bmu_coord, radius)
                self.weights[i, j] += learning_rate * influence * (input_vector - self.weights[i, j])

    def train(self, data, n_iterations=100, initial_lr=0.1, initial_radius=2.0):
        lr_decay = initial_lr / n_iterations
        radius_decay = initial_radius / n_iterations
        for iteration in range(n_iterations):
            current_lr = initial_lr - (iteration * lr_decay)
            current_radius = max(0.5, initial_radius - (iteration * radius_decay))
            sample_idx = np.random.randint(0, len(data))
            sample = data[sample_idx]
            bmu_coord = self.find_bmu(sample)
            self.update_weights(sample, bmu_coord, current_lr, current_radius)
        print('Treinamento concluído!')

    def predict(self, data):
        clusters = []
        for sample in data:
            bmu = self.find_bmu(sample)
            clusters.append(bmu)
        return np.array(clusters)

# Teste com 3 clusters
np.random.seed(42)
cluster1 = np.random.randn(20, 2) * 0.3 + [0, 0]
cluster2 = np.random.randn(20, 2) * 0.3 + [2, 2]
cluster3 = np.random.randn(20, 2) * 0.3 + [-2, 2]
data = np.vstack([cluster1, cluster2, cluster3])

print(f'Dados: {data.shape[0]} amostras, {data.shape[1]} dimensões')
som = SOM(map_width=3, map_height=3, input_dim=2)
print('Treinando SOM...')
som.train(data, n_iterations=100)

clusters = som.predict(data)
print(f'\nPrimeiras 10 atribuições de cluster:')
for i in range(10):
    print(f'  Amostra {i}: → posição {clusters[i]}')

In [ ]:
# Passo 6 — Monitoramento do treinamento (erro de quantização)
import numpy as np

def quantization_error(som, data):
    total_error = 0
    for sample in data:
        bmu = som.find_bmu(sample)
        error = np.linalg.norm(sample - som.weights[bmu])
        total_error += error
    return total_error / len(data)

np.random.seed(42)
cluster1 = np.random.randn(20, 2) * 0.3 + [0, 0]
cluster2 = np.random.randn(20, 2) * 0.3 + [2, 2]
cluster3 = np.random.randn(20, 2) * 0.3 + [-2, 2]
data = np.vstack([cluster1, cluster2, cluster3])

som_mon = SOM(map_width=5, map_height=5, input_dim=2)
errors = []
epochs = 20
samples_per_epoch = 50

for epoch in range(epochs):
    for _ in range(samples_per_epoch):
        sample = data[np.random.randint(0, len(data))]
        bmu = som_mon.find_bmu(sample)
        lr = 0.1 * (1 - epoch / epochs)
        radius = 2.0 * (1 - epoch / epochs)
        som_mon.update_weights(sample, bmu, lr, radius)
    error = quantization_error(som_mon, data)
    errors.append(error)
    print(f'Época {epoch:2d}: Erro = {error:.4f}')

print(f'\nErro inicial: {errors[0]:.4f}')
print(f'Erro final:   {errors[-1]:.4f}')
print(f'Redução: {(1 - errors[-1]/errors[0])*100:.1f}%')

In [ ]:
# Passo 7 — U-Matrix, Hit Map e Segmentação de Clientes
import numpy as np

def calculate_umatrix(som):
    umatrix = np.zeros((som.map_height, som.map_width))
    for i in range(som.map_height):
        for j in range(som.map_width):
            neighbors = []
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < som.map_height and 0 <= nj < som.map_width:
                    neighbors.append(som.weights[ni, nj])
            if neighbors:
                distances = [np.linalg.norm(som.weights[i,j] - n) for n in neighbors]
                umatrix[i, j] = np.mean(distances)
    return umatrix

def calculate_hit_map(som, data):
    hit_map = np.zeros((som.map_height, som.map_width))
    for sample in data:
        bmu = som.find_bmu(sample)
        hit_map[bmu] += 1
    return hit_map

# Segmentação de clientes: [idade, renda]
clientes = np.array([
    [20,1000],[22,1200],[21,1100],[23,1300],
    [35,5000],[38,5500],[40,6000],[36,5200],
    [65,8000],[68,8500],[70,9000],[67,8200],
], dtype=float)

# Normalizar para [0, 1]
clientes_norm = (clientes - clientes.min(0)) / (clientes.max(0) - clientes.min(0))

som_c = SOM(map_width=3, map_height=3, input_dim=2)
som_c.train(clientes_norm, n_iterations=200)

umatrix = calculate_umatrix(som_c)
hit_map = calculate_hit_map(som_c, clientes_norm)

print('U-MATRIX (valores altos = fronteiras):')
print(umatrix.round(3))
print('\nHIT MAP (frequência de ativação):')
print(hit_map)

clusters_c = som_c.predict(clientes_norm)
print('\nAtribuição de segmentos:')
for i, cl in enumerate(clientes):
    print(f'  Cliente {i+1} (idade={int(cl[0])}, renda={int(cl[1])}) → posição {clusters_c[i]}')

In [ ]:
# Passo 8 — Exercício: Quantização de Cores com SOM
import numpy as np

def train_color_som(colors, n_colors=16):
    map_size = int(np.sqrt(n_colors))
    colors_norm = colors.astype(float) / 255.0
    som = SOM(map_width=map_size, map_height=map_size, input_dim=3)
    som.train(colors_norm, n_iterations=500, initial_lr=0.3, initial_radius=float(map_size))
    return som

def reconstruct_image(som_model, original_colors):
    colors_norm = original_colors.astype(float) / 255.0
    quantized = []
    for color in colors_norm:
        bmu = som_model.find_bmu(color)
        quantized.append((som_model.weights[bmu] * 255).astype(np.uint8))
    return np.array(quantized)

print('Exercício: Quantização de Cores com SOM')
np.random.seed(42)

# Imagem sintética 10x10 RGB
img_fake = np.random.randint(0, 256, (10, 10, 3), dtype=np.uint8)
colors = img_fake.reshape(-1, 3)
print(f'Imagem: {img_fake.shape} → {colors.shape[0]} pixels RGB')
print(f'Cores únicas originais: {len(np.unique(colors, axis=0))}')

print('\nTreinando SOM para 16 cores...')
som_colors = train_color_som(colors, n_colors=16)

quantized_colors = reconstruct_image(som_colors, colors)
print(f'Cores únicas após quantização: {len(np.unique(quantized_colors, axis=0))}')
print('SOM reduziu o espaço de cores para 16 representantes!')

## Análise Crítica

**Aprendizado sem rótulos:** O SOM descobre estrutura nos dados sem nenhuma supervisão. Essa característica é valiosa quando os padrões "normais" não estão predefinidos, como em detecção de anomalias ou exploração de dados.

**Preservação topológica:** Amostras similares convergem para regiões próximas no mapa. A distância no mapa 2D reflete (aproximadamente) a distância no espaço original de alta dimensão. Clientes jovens/baixa renda ficam em posições opostas aos clientes seniores/alta renda.

**U-Matrix:** Posições com valores altos na U-Matrix indicam fronteiras naturais entre grupos. É a principal ferramenta visual para identificar clusters sem precisar definir o número deles a priori.

**Erro de quantização:** Mede quão bem o mapa representa os dados de entrada. A queda do erro ao longo das épocas confirma que o SOM está aprendendo uma representação cada vez mais fiel.


---


# Módulo 08 — Redes Neurais Sem Peso
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

As **redes neurais sem peso** (weightless neural networks) constituem uma classe fundamentalmente distinta: em vez de multiplicações em ponto flutuante com pesos, operam exclusivamente com **lógica booleana** e **tabelas de lookup em memória RAM**.

**Discriminador de RAM:** cada neurônio é uma memória endereçável. O vetor binário de entrada serve como endereço; a célula apontada armazena 0 ou 1. Treinar significa escrever 1 no endereço correspondente ao padrão apresentado.

**Classificação por votação:** vários discriminadores de RAM trabalham em paralelo. A saída final é o score — a soma dos discriminadores que respondem 1 para o padrão de consulta. Quanto mais discriminadores ativados, maior a confiança de que o padrão pertence à classe treinada.

**Capacidade de memória:** com $n$ bits de entrada, cada discriminador possui $2^n$ células. Isso cresce exponencialmente, limitando $n$ a valores modestos na prática. A vantagem é a velocidade: uma consulta é apenas um acesso à memória, sem nenhuma operação aritmética.

**Treinamento one-shot:** diferente de backpropagation, um único exemplo basta para ser memorizado — não há épocas.

In [ ]:
# Passo 2 — Portas lógicas booleanas
def AND(a, b):
    return 1 if (a == 1 and b == 1) else 0

def OR(a, b):
    return 1 if (a == 1 or b == 1) else 0

def NOT(x):
    return 1 - x

print('AND:')
for a in [0, 1]:
    for b in [0, 1]:
        print(f'  {a} AND {b} = {AND(a, b)}')

print('\nOR:')
for a in [0, 1]:
    for b in [0, 1]:
        print(f'  {a} OR {b} = {OR(a, b)}')

print('\nNOT:')
for x in [0, 1]:
    print(f'  NOT {x} = {NOT(x)}')

In [ ]:
# Passo 3 — Discriminador de RAM
import numpy as np

class RAMDiscriminator:
    def __init__(self, input_size):
        self.input_size = input_size
        self.memory_size = 2 ** input_size
        self.memory = np.zeros(self.memory_size, dtype=int)

    def address_to_index(self, pattern):
        index = 0
        for i, bit in enumerate(pattern):
            index += bit * (2 ** i)
        return int(index)

    def train(self, pattern):
        index = self.address_to_index(pattern)
        self.memory[index] = 1

    def discriminate(self, pattern):
        index = self.address_to_index(pattern)
        return self.memory[index]

ram = RAMDiscriminator(input_size=3)
print('Memória inicial:', ram.memory)

pattern1 = np.array([1, 0, 1])
ram.train(pattern1)
print(f'\nTreinado com {pattern1}')
print('Memória atualizada:', ram.memory)
print(f'\nConsultar {pattern1}: {ram.discriminate(pattern1)}')
print(f'Consultar {np.array([0,1,0])}: {ram.discriminate(np.array([0,1,0]))}')

In [ ]:
# Passo 4 — Rede Sem Peso completa + Classificação de caracteres
import numpy as np

class WeightlessNeuralNetwork:
    def __init__(self, input_size, num_discriminators=None):
        self.input_size = input_size
        if num_discriminators is None:
            num_discriminators = input_size
        self.num_discriminators = num_discriminators
        self.rams = [RAMDiscriminator(input_size) for _ in range(num_discriminators)]

    def train(self, patterns):
        for pattern in patterns:
            for ram in self.rams:
                ram.train(pattern)

    def predict(self, pattern):
        score = 0
        for ram in self.rams:
            score += ram.discriminate(pattern)
        return score

    def classify(self, pattern, threshold=0.5):
        score = self.predict(pattern)
        return 1 if score >= (self.num_discriminators * threshold) else 0

# Caracteres 3x3 (9 bits)
A = np.array([0, 1, 0, 1, 0, 1, 1, 1, 1])
B = np.array([1, 1, 0, 1, 1, 0, 1, 1, 1])
C = np.array([0, 1, 1, 1, 0, 0, 1, 1, 1])

wnn_A = WeightlessNeuralNetwork(input_size=9); wnn_A.train([A])
wnn_B = WeightlessNeuralNetwork(input_size=9); wnn_B.train([B])
wnn_C = WeightlessNeuralNetwork(input_size=9); wnn_C.train([C])

print('CLASSIFICAÇÃO DE CARACTERES (3x3 pixels):')
for char, label in [(A,'A'),(B,'B'),(C,'C')]:
    scores = {'A': wnn_A.predict(char), 'B': wnn_B.predict(char), 'C': wnn_C.predict(char)}
    predicted = max(scores, key=scores.get)
    print(f'  Char {label}: scores={scores} → Predito: {predicted} | OK: {predicted==label}')

print('\nTeste com exemplo geral (padrões positivos):')
positive_patterns = [np.array([1,1,1,1]), np.array([1,0,1,1]), np.array([1,1,0,1])]
wnn = WeightlessNeuralNetwork(input_size=4)
wnn.train(positive_patterns)
for pat in [np.array([1,1,1,1]), np.array([0,0,0,0])]:
    score = wnn.predict(pat)
    print(f'  {pat}: score={score}/{wnn.num_discriminators} → {"POSITIVO" if wnn.classify(pat) else "NEGATIVO"}')

## Análise Crítica

**One-shot learning:** O ponto mais distintivo das redes sem peso é que uma única apresentação de cada padrão é suficiente. Não existe convergência iterativa — o padrão é gravado diretamente na RAM em O(1).

**Classificação exata vs. generalização:** Para padrões idênticos aos treinados, o score é máximo. Para padrões diferentes, o score é 0 — a rede não generaliza para variações. Isso é exatamente o oposto do que o MLP faz via gradiente.

**Crescimento exponencial de memória:** Com $n$ bits de entrada, cada discriminador precisa de $2^n$ células. Para entradas de imagem (mesmo pequenas, como 28x28 = 784 bits) isso seria computacionalmente inviável. A solução prática é particionar a entrada em blocos menores.

**Aplicações ideais:** Sistemas onde velocidade é crítica e os padrões são conhecidos com antecedência — controle em tempo real, reconhecimento de padrões binários fixos, sistemas embarcados sem unidade de ponto flutuante.


---


# Módulo 09 — Redes ART (Adaptive Resonance Theory)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Desenvolvidas por Stephen Grossberg e Gail Carpenter, as redes **ART** enfrentam o dilema central do aprendizado neural:

- **Plasticidade:** capacidade de assimilar novos padrões
- **Estabilidade:** capacidade de não destruir o que foi aprendido anteriormente

Redes convencionais sofrem de **catastrophic forgetting**: aprender uma nova tarefa apaga padrões anteriores. A ART resolve isso com um **parâmetro de vigilância $\rho$** que controla quando criar uma nova categoria versus atualizar uma existente.

**Critério de vigilância (ART-1 — entradas binárias):**

$$\frac{|\mathbf{x} \cap \mathbf{w}|}{|\mathbf{x}|} \geq \rho$$

Se a similaridade entre a entrada $\mathbf{x}$ e o protótipo da categoria $\mathbf{w}$ atingir o limiar $\rho$, ocorre **ressonância**: a categoria é refinada pela interseção. Caso contrário, uma nova categoria é criada.

**Efeito de $\rho$:** valores altos criam categorias muito específicas (muitas classes finas); valores baixos agrupam padrões diferentes numa mesma categoria (menos classes grosseiras).

In [ ]:
# Passo 5 — Classe ART-1 completa
import numpy as np

class ART1:
    def __init__(self, input_size, vigilance=0.7):
        self.input_size = input_size
        self.vigilance = vigilance
        self.categories = []
        self.num_categories = 0

    def compute_similarity(self, pattern, weights):
        intersection = np.sum(pattern * weights)
        pattern_sum = np.sum(pattern)
        return intersection / pattern_sum if pattern_sum > 0 else 0

    def train(self, pattern):
        if self.num_categories == 0:
            self.categories.append(pattern.copy())
            self.num_categories = 1
            return (0, False)

        tried_categories = set()
        while len(tried_categories) < self.num_categories:
            for i in range(self.num_categories):
                if i not in tried_categories:
                    category = i
                    tried_categories.add(i)
                    break
            similarity = self.compute_similarity(pattern, self.categories[category])
            if similarity >= self.vigilance:
                self.categories[category] = pattern * self.categories[category]
                return (category, False)

        self.categories.append(pattern.copy())
        self.num_categories += 1
        return (self.num_categories - 1, True)

    def predict(self, pattern):
        if self.num_categories == 0:
            return None
        best_cat = 0
        best_sim = self.compute_similarity(pattern, self.categories[0])
        for i in range(1, self.num_categories):
            sim = self.compute_similarity(pattern, self.categories[i])
            if sim > best_sim:
                best_sim = sim
                best_cat = i
        return best_cat if best_sim >= self.vigilance else None

art = ART1(input_size=6, vigilance=0.6)
print('APRENDIZADO INCREMENTAL COM ART-1')
print('='*40)

data_stream = [
    np.array([1,1,1,0,0,0]),
    np.array([1,1,0,0,0,0]),
    np.array([1,1,1,1,0,0]),
    np.array([0,0,0,1,1,1]),
    np.array([0,0,0,1,1,0]),
    np.array([1,0,0,0,0,0]),
]

for i, pattern in enumerate(data_stream):
    category, was_new = art.train(pattern)
    novo_str = ' [NOVA]' if was_new or i == 0 else ''
    print(f'Amostra {i+1}: {pattern} → Categoria {category}{novo_str}')

print(f'\nTotal de categorias: {art.num_categories}')
print('Rede aprendeu incrementalmente sem esquecer padrões anteriores!')

In [ ]:
# Passo 6 — Clustering de dados streaming com 3 clusters naturais
import numpy as np

np.random.seed(42)
art2 = ART1(input_size=4, vigilance=0.75)

stream = []
# Cluster A: [1,1,0,0] com variações
for _ in range(5):
    pattern = np.array([1,1,0,0])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

# Cluster B: [0,0,1,1] com variações
for _ in range(5):
    pattern = np.array([0,0,1,1])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

# Cluster C: [1,0,1,0] com variações
for _ in range(5):
    pattern = np.array([1,0,1,0])
    if np.random.rand() > 0.7:
        pattern = pattern.copy()
        pattern[np.random.randint(0,4)] = 1
    stream.append(pattern)

for i, pattern in enumerate(stream):
    category, was_new = art2.train(pattern)
    novo_str = ' [NOVA CATEGORIA]' if was_new or i == 0 else ''
    print(f'Stream {i+1:2d}: {pattern} → Cat {category}{novo_str}')

print(f'\nCategorias descobertas: {art2.num_categories}')
print('\nAgrupamentos finais:')
for i in range(art2.num_categories):
    print(f'  Categoria {i}: {art2.categories[i]}')

## Análise Crítica

**Solução ao catastrophic forgetting:** A ART-1 é a única arquitetura deste portfólio que genuinamente preserva padrões anteriores ao assimilar novos. A criação de novas categorias protege as memórias existentes — um padrão suficientemente diferente nunca sobrescreve uma categoria já formada.

**Papel da vigilância $\rho$:** valores altos (ex: 0.9) geram muitas categorias finas — cada variação pequena cria uma nova classe (análogo ao overfitting). Valores baixos (ex: 0.3) agrupam padrões muito diferentes numa mesma categoria (análogo ao underfitting).

**Ordem de apresentação importa:** diferente do SOM que processa todos os dados em cada época, a ART-1 aprende de forma estritamente incremental. A ordem em que os padrões chegam influencia quais categorias são formadas e como são refinadas.

**Limitação de escalabilidade:** a ART-1 trabalha apenas com entradas binárias. Extensões como ART-2 e Fuzzy ART lidam com entradas contínuas, mas aumentam a complexidade do mecanismo de vigilância.


---


# Módulo 10 — Redes BAM (Bidirectional Associative Memory)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Criada por Bart Kosko em 1988, a **BAM** estende a memória associativa linear (Módulo 05) para suportar recuperação **em ambas as direções**: dado $X$ recupera $Y$, e dado $Y$ recupera $X$ de volta.

A **regra de aprendizado de Kosko** constrói duas matrizes simétricas a partir dos pares de treinamento:

$$W = \sum_{k} \mathbf{y}_k \mathbf{x}_k^T \qquad W^T = \sum_{k} \mathbf{x}_k \mathbf{y}_k^T$$

A simetria é garantida matematicamente: $W^T$ é exatamente a transposta de $W$. A recuperação alterna entre as duas direções até convergir:

$$Y' = \text{sign}(W^T X) \qquad X' = \text{sign}(W Y')$$

**Capacidade:** a BAM armazena com confiabilidade $\min(N, M)$ pares, onde $N$ e $M$ são os tamanhos dos vetores $X$ e $Y$ respectivamente. Para pares ortogonais a recuperação é exata; para pares correlacionados pode haver interferência.

In [ ]:
# Passo 3 — Função train_bam
import numpy as np

def train_bam(pairs):
    if len(pairs) == 0:
        raise ValueError('Pelo menos um par é necessário')
    size_x = len(pairs[0][0])
    size_y = len(pairs[0][1])
    W = np.zeros((size_x, size_y))
    W_transpose = np.zeros((size_y, size_x))
    for X, Y in pairs:
        X = np.array(X).reshape(-1, 1)
        Y = np.array(Y).reshape(-1, 1)
        W += np.dot(X, Y.T)
        W_transpose += np.dot(Y, X.T)
    return W, W_transpose

pairs = [
    (np.array([1, 0, 1]), np.array([1, 0])),
    (np.array([0, 1, 0]), np.array([0, 1])),
    (np.array([1, 1, 0]), np.array([1, 1])),
]

W, W_t = train_bam(pairs)
print('Matriz W (X → Y):')
print(W)
print('\nMatriz W^T (Y → X):')
print(W_t)
print(f'\nW é transposta de W^T? {np.allclose(W, W_t.T)}')

In [ ]:
# Passo 4 — Classe BAM completa com recuperação bidirecional
import numpy as np

class BAM:
    def __init__(self, size_x, size_y):
        self.size_x = size_x
        self.size_y = size_y
        self.W = np.zeros((size_x, size_y))
        self.W_transpose = np.zeros((size_y, size_x))

    def train(self, pairs):
        self.W = np.zeros((self.size_x, self.size_y))
        self.W_transpose = np.zeros((self.size_y, self.size_x))
        for X, Y in pairs:
            X = np.array(X).reshape(-1, 1)
            Y = np.array(Y).reshape(-1, 1)
            self.W += np.dot(X, Y.T)
            self.W_transpose += np.dot(Y, X.T)

    def recall_x_to_y(self, X):
        X = np.array(X).reshape(-1, 1)
        net = np.dot(self.W.T, X)
        return np.where(net.flatten() >= 0, 1, 0)

    def recall_y_to_x(self, Y):
        Y = np.array(Y).reshape(-1, 1)
        net = np.dot(self.W, Y)
        return np.where(net.flatten() >= 0, 1, 0)

bam = BAM(size_x=3, size_y=2)
pairs = [
    (np.array([1, 0, 1]), np.array([1, 0])),
    (np.array([0, 1, 0]), np.array([0, 1])),
    (np.array([1, 1, 0]), np.array([1, 1])),
]
bam.train(pairs)

print('MEMÓRIA ASSOCIATIVA BIDIRECIONAL')
print('Associações treinadas:')
for X, Y in pairs:
    print(f'  {X} ↔ {Y}')

print('\nRecuperação X → Y:')
for X, Y in pairs:
    rec_Y = bam.recall_x_to_y(X)
    print(f'  {X} → {rec_Y} (esperado: {Y}) | OK: {np.array_equal(rec_Y, Y)}')

print('\nRecuperação Y → X:')
for X, Y in pairs:
    rec_X = bam.recall_y_to_x(Y)
    print(f'  {Y} → {rec_X} (esperado: {X}) | OK: {np.array_equal(rec_X, X)}')

In [ ]:
# Passo 5 — Teste completo de recuperação (4 pares)
import numpy as np

bam4 = BAM(size_x=4, size_y=4)
pairs4 = [
    (np.array([1,0,0,0]), np.array([1,0,0,0])),
    (np.array([0,1,0,0]), np.array([0,1,0,0])),
    (np.array([0,0,1,0]), np.array([0,0,1,0])),
    (np.array([0,0,0,1]), np.array([0,0,0,1])),
]
bam4.train(pairs4)
colors = ['Vermelho','Verde','Azul','Amarelo']

print('Recuperação X → Y (Nome → Cor):')
for i, (X, Y) in enumerate(pairs4):
    rec = bam4.recall_x_to_y(X)
    match = np.array_equal(rec, Y)
    print(f'  {X} → {rec} ({colors[i]}): {"✓" if match else "✗"}')

print('\nRecuperação Y → X (Cor → Nome):')
for i, (X, Y) in enumerate(pairs4):
    rec = bam4.recall_y_to_x(Y)
    match = np.array_equal(rec, X)
    print(f'  {Y} → {rec} ({colors[i]}): {"✓" if match else "✗"}')

print('\nTeste com padrão ruidoso:')
noisy_X = np.array([1,0,0,1])  # Vermelho com 1 bit trocado
rec_Y = bam4.recall_x_to_y(noisy_X)
print(f'  Entrada ruidosa: {noisy_X}')
print(f'  Recuperado Y:    {rec_Y}')

## Análise Crítica

**Bidirecionalidade por transposição:** A característica central da BAM é matematicamente elegante — a mesma informação codificada em $W$ serve para $Y \to X$, e sua transposta $W^T$ serve para $X \to Y$. Não há custo adicional de armazenamento para suportar ambas as direções.

**Comparação com o Módulo 05:** A memória associativa de Hebb é unidirecional ($X \to Y$ apenas). A BAM adiciona a direção inversa sem custo teórico extra — os mesmos pesos, apenas usados transpostos.

**Ortogonalidade e capacidade:** Para pares ortogonais (como os vetores one-hot do exercício 4x4) a recuperação é perfeita em ambas as direções. Com pares correlacionados a interferência cruzada aumenta e a capacidade efetiva cai.

**Recuperação iterativa:** Embora a implementação aqui seja de passo único, a BAM pode ser executada de forma iterativa — $X \to Y \to X' \to Y' \to \ldots$ — até convergir num ponto fixo. Esse processo é garantido de convergir quando $W$ é simétrica, o que é sempre o caso.


---


# Módulo 11 — Redes Convolucionais (CNN)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

As **CNNs** revolucionaram a visão computacional ao substituir a abordagem do MLP — que "achata" a imagem em um vetor perdendo a estrutura espacial — por operações que **preservam a localidade 2D** dos pixels.

**Operação de convolução:** um filtro $K$ de tamanho $k \times k$ desliza sobre a imagem $I$:

$$(I * K)[i,j] = \sum_{m}\sum_{n} I[i+m, j+n] \cdot K[m,n]$$

Cada filtro detecta um padrão específico (borda, textura, forma) em qualquer posição da imagem. Múltiplos filtros são aplicados em paralelo, gerando um conjunto de **feature maps**.

**Max Pooling:** reduz a resolução espacial mantendo apenas o valor máximo de cada região, introduzindo invariância translacional:

$$P[i,j] = \max_{m,n \in \text{pool}} F[i \cdot s + m,\, j \cdot s + n]$$

**Parâmetros compartilhados:** um filtro $3 \times 3$ tem apenas $9$ parâmetros, independentemente do tamanho da imagem. O mesmo filtro varre toda a imagem — isso reduz drasticamente o número de parâmetros comparado ao MLP equivalente.

**Hierarquia de representações:** camadas iniciais aprendem bordas e texturas; camadas mais profundas combinam esses elementos em formas e objetos complexos.

In [ ]:
# Passo 2-3 — Convolução e Max Pooling manuais
import numpy as np

def convolve2d(image, kernel):
    i_h, i_w = image.shape
    k_h, k_w = kernel.shape
    o_h = i_h - k_h + 1
    o_w = i_w - k_w + 1
    output = np.zeros((o_h, o_w))
    for i in range(o_h):
        for j in range(o_w):
            region = image[i:i+k_h, j:j+k_w]
            output[i, j] = np.sum(region * kernel)
    return output

def max_pool2d(feature_map, pool_size=2):
    h, w = feature_map.shape
    out_h = h // pool_size
    out_w = w // pool_size
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = feature_map[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(region)
    return output

# Demonstração: imagem com borda vertical
image = np.array([
    [1,1,0,0,0],
    [1,1,0,0,0],
    [1,1,0,0,0],
    [1,1,0,0,0],
    [1,1,0,0,0]
])
kernel_borda = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])

print('Imagem (5x5):\n', image)
print('\nKernel detector de borda vertical (3x3):\n', kernel_borda)

fm = convolve2d(image, kernel_borda)
print('\nFeature map após convolução (3x3):\n', fm)

# Max Pooling
fm4 = np.array([[1,3,2,4],[5,2,1,0],[3,6,2,1],[1,4,5,2]])
pooled = max_pool2d(fm4, pool_size=2)
print('\nMax pooling 2x2 de [[1,3,2,4],[5,2,1,0],[3,6,2,1],[1,4,5,2]]:')
print(pooled)
print('4x4 → 2x2: redução de 75% dos dados mantendo valores máximos')

In [ ]:
# Passo 5 — CNN com TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import layers, models

def create_simple_cnn(input_shape, num_classes):
    model = models.Sequential([
        # Primeiro bloco convolucional
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        # Segundo bloco convolucional
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        # Classificador
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print('Criando CNN para MNIST (28x28x1, 10 classes)...')
model = create_simple_cnn(input_shape=(28,28,1), num_classes=10)
model.summary()
print(f'\nTotal de parâmetros: {model.count_params():,}')
print('Parâmetros compartilhados via filtros = muito mais eficiente que MLP!')

In [ ]:
# Passo 6 — Treinamento com MNIST
import tensorflow as tf
from tensorflow.keras import datasets
from tensorflow.keras.utils import to_categorical

print('Carregando MNIST...')
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

train_images = train_images.reshape((-1,28,28,1)).astype('float32') / 255.0
test_images  = test_images.reshape((-1,28,28,1)).astype('float32') / 255.0
train_labels_cat = to_categorical(train_labels, 10)
test_labels_cat  = to_categorical(test_labels, 10)

print(f'Treino: {train_images.shape[0]} imagens | Teste: {test_images.shape[0]} imagens')
print(f'Formato: {train_images.shape[1:]}')

print('\nTreinando CNN (5 épocas)...')
history = model.fit(
    train_images, train_labels_cat,
    epochs=5,
    batch_size=128,
    validation_data=(test_images, test_labels_cat),
    verbose=1
)

test_loss, test_acc = model.evaluate(test_images, test_labels_cat, verbose=0)
print(f'\nAcurácia final no teste: {test_acc*100:.2f}%')

In [ ]:
# Passo 7 — Visualização dos filtros aprendidos
import numpy as np

filters, biases = model.layers[0].get_weights()
print(f'Filtros da 1ª camada Conv2D: {filters.shape}')
print(f'  → 32 filtros de 3x3x1 (3x3 pixels, 1 canal)')

f_min, f_max = filters.min(), filters.max()
filters_norm = (filters - f_min) / (f_max - f_min)

print('\nEstatísticas dos primeiros 8 filtros:')
for i in range(8):
    f = filters[:,:,0,i]
    print(f'  Filtro {i+1:2d}: min={f.min():.3f}, max={f.max():.3f}, std={f.std():.3f}')

print('\nFiltros com alto contraste detectam bordas e texturas.')
print('Filtros uniformes detectam regiões homogêneas.')
print('Camadas mais profundas combinam esses padrões em formas complexas.')

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < 8:
            ax.imshow(filters_norm[:,:,0,i], cmap='RdBu')
            ax.set_title(f'Filtro {i+1}')
            ax.axis('off')
    plt.suptitle('Filtros aprendidos — 1ª camada Conv2D')
    plt.tight_layout()
    plt.savefig('filtros_cnn.png', dpi=80, bbox_inches='tight')
    plt.show()
    print('Visualização salva em filtros_cnn.png')
except Exception as e:
    print(f'(Visualização matplotlib indisponível: {e})')

## Análise Crítica

**CNN vs. MLP em imagens:** O MLP trata cada pixel como uma feature independente, ignorando a relação de vizinhança. A CNN, por sua vez, aplica filtros locais que exploram explicitamente a estrutura espacial — bordas em $(i,j)$ têm relação direta com o que está em $(i+1,j)$.

**Parâmetros compartilhados:** Um filtro $3 \times 3$ com 9 parâmetros é aplicado nas 676 posições de uma imagem $28 \times 28$. Um MLP equivalente precisaria de 676 parâmetros por neurônio na primeira camada, sem nenhuma reutilização. O compartilhamento não só economiza parâmetros como força o modelo a aprender detectores de padrões genéricos.

**Hierarquia de abstrações:** Filtros da primeira camada respondem a bordas e gradientes de cor. Filtros da segunda camada combinam esses elementos em formas geométricas. Camadas ainda mais profundas reconhecem partes de objetos. Essa hierarquia emergente é a principal razão pela qual CNNs profundas superam modelos rasos.

**Invariância translacional via pooling:** O Max Pooling reduz a sensibilidade à posição exata de um padrão — um filtro que detecta uma borda à esquerda produz a mesma resposta se a borda se deslocar alguns pixels, depois do pooling.